# Getting Started, Chapter 2 -- Sampling & intervention

This chapter takes the SIR network from Chapter 1 and shows how to draw
parameter sets from it, how to sample only the part you need, and how to run
*interventions* -- fixing a node and letting the change propagate downstream.

In [ ]:
from sims_pars import (bayes_net_from_script, sample, sample_minimally,
                       sample_chromosome)
import numpy as np

SIR_SCRIPT = '''
PCore SIR {
    pop = 1000
    beta  ~ unif(1.0, 3.0)
    gamma ~ unif(0.2, 1.0)
    r0 = beta / gamma
    cases ~ binom(pop, prev0)
}
'''
bn = bayes_net_from_script(SIR_SCRIPT, strict=True)

## A single draw

`sample(bn, cond)` walks the graph in topological order and returns a dict with
a value for every node. Exogenous nodes (`prev0`) must be given in `cond`.

In [ ]:
sample(bn, {'prev0': 0.02})

In [ ]:
# each call redraws the random nodes
import pandas as pd
draws = pd.DataFrame(sample(bn, {'prev0': 0.02}) for _ in range(1000))
draws[['beta', 'gamma', 'r0', 'cases']].describe().round(3)

## Log-probability and the `Chromosome`

`sample_chromosome` returns a `Chromosome`: one realised parameter set plus its
log-probability (the sum of `logpdf` over the random nodes).

In [ ]:
ch = sample_chromosome(bn, {'prev0': 0.02})
print(dict(ch))
print('log p =', ch.LogProb)

## Sampling only what you need

If you only care about some outputs, `sample_minimally` draws just their
ancestors. Here we ask for `r0` and get back only `beta` and `gamma`; `cases`
and its `binom` draw are skipped entirely.

In [ ]:
sinks, mediators = sample_minimally(bn, included=['r0'], cond={'prev0': 0.02})
print('requested :', sinks)
print('mediators :', mediators)

## Intervention: the do-operator

`Chromosome.impulse({node: value}, bn)` fixes a node and re-renders every
descendant from the new value -- this is `do(gamma = ...)`, not conditioning.
The log-probability is invalidated because the draw is no longer from the prior.

In [ ]:
ch = sample_chromosome(bn, {'prev0': 0.02})
before = dict(ch)

ch.impulse({'gamma': 0.25}, bn)     # do(gamma = 0.25)
after = dict(ch)

pd.DataFrame({'before': before, 'after do(gamma=0.25)': after})

`beta` is unchanged (it is not downstream of `gamma`), while `r0` jumps to
`beta / 0.25`. Passing `None` re-draws a node from its prior instead:

In [ ]:
ch.impulse({'gamma': None}, bn)     # resample gamma from unif(0.2, 1.0)
dict(ch)

### Observational vs interventional

Compare the distribution of `r0` when we *observe* a low `gamma` (rejection
sampling) with when we *set* `gamma` to the same range with `impulse`.

In [ ]:
np.random.seed(1)
obs = pd.Series(
    s['r0'] for s in (sample(bn, {'prev0': 0.02}) for _ in range(20000))
    if s['gamma'] < 0.3
)

inter = []
for _ in range(2000):
    c = sample_chromosome(bn, {'prev0': 0.02})
    c.impulse({'gamma': np.random.uniform(0.2, 0.3)}, bn)
    inter.append(c['r0'])
inter = pd.Series(inter)

pd.DataFrame({'observe gamma<0.3': obs.describe(),
              'do(gamma~U(0.2,0.3))': inter.describe()}).round(3)

Here the two match, because `gamma` has no parents: with nothing upstream to
*explain away*, conditioning on `gamma < 0.3` and forcing it into that range
are the same operation. They diverge once the intervened node has parents --
then `do(x = v)` cuts the incoming edges (breaking any confounding) while
conditioning does not. That is the whole reason the do-operator exists.

## Many draws

A parameter sweep is just a loop over `sample` (or `sample_chromosome` if you
also want each draw's log-probability). Reseeding NumPy makes it reproducible.

In [ ]:
np.random.seed(0)
prior = pd.DataFrame(sample(bn, {'prev0': 0.02}) for _ in range(5000))
prior[['beta', 'gamma', 'r0']].agg(['mean', 'std', 'min', 'max']).round(3)

For **structured** simulation -- a hierarchy of actors, parameters that are
fixed at one level and resampled at another, deterministic seeding of whole
sub-models -- `sims-pars` has a `SimulationCore` layer (`as_simulation_core`,
`NodeSet`). It is worth a chapter of its own; see the
[Simulation core](../concepts/simulation-core.md) concept page and the
**Tutorials** section.

---
**Next:** [Chapter 3 -- Fitting a model](GettingStarted03_Fitting a model.ipynb)